# S04 toy — structured generation: schema, validator, retry loop

A small-device repair shop turns customer intake notes into work orders. The shop's scheduling system eats only JSON that validates against the work-order schema — so the model's output crosses a boundary, and everything that crosses a boundary gets validated. No network, no keys: the "model" is a plain function you can read end to end.

**How to use:** run cells in order. For each experiment, write your prediction as a comment *before* running. The gap between prediction and result is the lesson.

## The intake notes

Five briefs, in escalating difficulty: one clear, one vague, one overloaded, one in Spanish, and one whose device is *not in the schema's enum*. Keep that last one in mind.

In [ ]:
BRIEFS = [
    # 0: clear
    "Hi, I'm Dana. My laptop won't charge — the port feels loose. No hurry, "
    "I have a spare machine. Pretty sure the charging port needs replacing.",
    # 1: vague
    "hey it's marco, tablet acting weird?? screen does the flicker thing "
    "sometimes. whenever you guys have time tbh",
    # 2: overloaded
    "This is Priya at Northwind. Our office phone — the main line — died this "
    "morning, the backup console overheats, and the phone battery looks swollen. "
    "I need the phone back by tomorrow 8am for a client demo or we lose the account.",
    # 3: Spanish
    "Hola, soy Lucía. Mi laptop se apaga sola a los diez minutos, creo que es "
    "la batería. La necesito para el viernes, gracias.",
    # 4: device outside the schema's enum
    "Name's Theo. My e-bike's display goes dark on hills — the shop down the "
    "road said it's the controller. It's my commute, so it's urgent for me.",
]

## The contract: schema + hand-rolled validator

The schema is the shop's data model. The validator is hand-rolled over a JSON-Schema subset — `type`, `required`, `properties`, `enum` — because the checks *are* the lesson; the `jsonschema` library is these same checks with more keywords. Note what `required` says: without those five fields, the scheduler cannot proceed.

In [ ]:
WORK_ORDER_SCHEMA = {
    "type": "object",
    "required": ["customer", "device", "priority", "symptoms", "parts_mentioned"],
    "properties": {
        "customer": {"type": "string"},
        "device": {"type": "string", "enum": ["laptop", "phone", "tablet", "console"]},
        "priority": {"type": "string", "enum": ["low", "normal", "urgent"]},
        "symptoms": {"type": "array"},
        "parts_mentioned": {"type": "array"},
        "notes": {"type": "string"},         # optional: not in `required`
        "estimate_usd": {"type": "number"},  # optional
    },
}

In [ ]:
import json

TYPE_CHECKS = {
    "object":  lambda v: isinstance(v, dict),
    "array":   lambda v: isinstance(v, list),
    "string":  lambda v: isinstance(v, str),
    # bool IS an int in Python — real validators exclude it explicitly, so do we
    "number":  lambda v: isinstance(v, (int, float)) and not isinstance(v, bool),
    "integer": lambda v: isinstance(v, int) and not isinstance(v, bool),
    "boolean": lambda v: isinstance(v, bool),
}

def validate(instance, schema, path="$"):
    """JSON-Schema subset: type / required / properties / enum.
    Returns a list of human-readable errors ([] means valid)."""
    if "type" in schema and not TYPE_CHECKS[schema["type"]](instance):
        return [f"{path}: expected {schema['type']}, got {json.dumps(instance)[:50]}"]
    errors = []
    if "enum" in schema and instance not in schema["enum"]:
        errors.append(f"{path}: {json.dumps(instance)} is not one of {schema['enum']}")
    if schema.get("type") == "object":
        for field in schema.get("required", []):
            if field not in instance:
                errors.append(f"{path}: missing required field {field!r}")
        for field, sub in schema.get("properties", {}).items():
            if field in instance:
                errors += validate(instance[field], sub, f"{path}.{field}")
    return errors

The validator gets the fixture-invariant treatment (S02): a known-good instance must pass, a known-bad one must fail *for the expected reason*. A validator that can't fail asserts nothing.

In [ ]:
_good = {"customer": "Dana", "device": "laptop", "priority": "low",
         "symptoms": ["won't charge"], "parts_mentioned": ["charging port"]}
_bad = {**_good, "priority": "high"}   # "high" is not in the enum

assert validate(_good, WORK_ORDER_SCHEMA) == []
_errs = validate(_bad, WORK_ORDER_SCHEMA)
assert len(_errs) == 1 and "$.priority" in _errs[0], _errs
print("validator self-test: good passes, bad fails for the expected reason")
print(" ", _errs[0])

## The mock model

A plain function with a per-brief *script* of raw outputs, indexed by attempt number (counted from the message list — the API is stateless, so the mock is too). Read `MOCK_OUTPUTS` before running anything: the model's entire behavior is on the page. Attempt 0 is what prompt-and-pray gets you; later attempts are what the model does *after seeing your error feedback*.

In [ ]:
def _wo(customer, device, priority, symptoms, parts, **extra):
    d = {"customer": customer, "device": device, "priority": priority,
         "symptoms": symptoms, "parts_mentioned": parts}
    d.update(extra)
    return json.dumps(d, ensure_ascii=False)

_b1 = _wo("Marco", "tablet", "low", ["screen flickers intermittently"], [])
_b2_missing = json.dumps({
    "customer": "Priya (Northwind)", "device": "phone",
    "symptoms": ["office phone dead", "backup console overheats", "swollen battery"],
    "parts_mentioned": ["battery"],
    "notes": "Needs the phone back by tomorrow 8am — client demo."}, ensure_ascii=False)
_b4 = _wo("Theo", "e-bike", "urgent", ["display goes dark on hills"], ["controller"])

MOCK_OUTPUTS = {
    0: [_wo("Dana", "laptop", "low", ["won't charge", "loose charging port"],
            ["charging port"], notes="Has a spare machine; no rush.")],
    1: [f"Sure, here's the work order!\n```json\n{_b1}\n```\nLet me know if you need anything else.",
        _b1],
    2: [_b2_missing,
        _wo("Priya (Northwind)", "phone", "urgent",
            ["office phone dead", "backup console overheats", "swollen battery"],
            ["battery"], notes="Needs the phone back by tomorrow 8am — client demo.")],
    3: [_wo("Lucía", "portátil", "alta", ["se apaga sola a los diez minutos"], ["batería"]),
        _wo("Lucía", "laptop", "normal", ["shuts off by itself after ~10 min"], ["battery"],
            notes="Needs it by Friday.")],
    4: [_b4, _b4, _b4, _b4],   # the model keeps reporting the device it actually sees
}

def mock_model(messages):
    """Stateless stand-in for POST /v1/chat/completions. Fully inspectable."""
    brief_idx = next(i for i, b in enumerate(BRIEFS) if b in messages[0]["content"])
    attempt = sum(1 for m in messages if m["role"] == "assistant")
    script = MOCK_OUTPUTS[brief_idx]
    raw = script[min(attempt, len(script) - 1)]
    msg = {"role": "assistant", "content": raw, "tool_calls": []}
    return {"choices": [{"message": msg}], "usage": {"total_tokens": 30}}

## Experiment 1 — prompt-and-pray

One shot per brief, no retries: the raw output is parsed and validated, and we tally what survives. This is the naive baseline — S02's reasoning applies: you measure machinery against the status quo, and the status quo is "ask nicely, hope for JSON."

**Predict first:** for each of the five briefs — parse error, validation error, or pass? Fill the dict, then run the solution cell.

In [ ]:
# YOUR PREDICTION — fill before running the solution cell.
# One of "parse-error", "invalid", or "pass" per brief index.
PREDICTION = {}   # e.g. {0: "pass", 1: "parse-error", 2: "invalid", ...}

In [ ]:
# SOLUTION — naive single-shot extraction over all five briefs.
PROMPT = ("You are the intake clerk of a small-device repair shop. Read the "
          "customer note and reply with ONE raw JSON object matching the "
          "work-order schema: customer, device, priority, symptoms, "
          "parts_mentioned (notes, estimate_usd optional). No prose, no fences."
          "\n\nCustomer note:\n")

def classify(brief):
    msg = mock_model([{"role": "user", "content": PROMPT + brief}])["choices"][0]["message"]
    try:
        data = json.loads(msg["content"])
    except json.JSONDecodeError as exc:
        return "parse-error", str(exc).split(":")[0]
    errors = validate(data, WORK_ORDER_SCHEMA)
    return ("pass", "") if not errors else ("invalid", errors[0])

print(f"{'brief':<6} {'outcome':<12} first error / detail")
n_pass = 0
for i, brief in enumerate(BRIEFS):
    outcome, detail = classify(brief)
    n_pass += outcome == "pass"
    print(f"{i:<6} {outcome:<12} {detail[:70]}")
print(f"\nnaive baseline: {n_pass}/{len(BRIEFS)} valid "
      f"— you predicted {dict(PREDICTION) or 'nothing'}")

## Experiment 2 — validate-and-retry (you build the loop)

Same machinery as S01: the error goes *into the messages* — a `PARSE ERROR:` or `VALIDATION ERRORS:` user message — and the model re-asks. Capped, always: an uncapped retry loop is a non-terminating agent.

**Predict first:** which briefs converge, and within how many attempts? Which one *can't*, and why? Write your loop in the attempt cell, then run the solution.

In [ ]:
# YOUR ATTEMPT — implement the retry loop before opening the solution cell.
# Contract: call the model; parse; validate; on failure append the error as a
# user message (errors are messages — S01) and retry; cap at max_attempts.
def extract_with_retry_mine(brief, max_attempts=4):
    return None, [], 0   # replace with your loop

print("attempt cell ran — write your loop, then compare with the solution below")

In [ ]:
# SOLUTION — the validate-and-retry loop.
def extract_with_retry(brief, schema=WORK_ORDER_SCHEMA, max_attempts=4):
    messages = [{"role": "user", "content": PROMPT + brief}]
    for attempt in range(1, max_attempts + 1):
        msg = mock_model(messages)["choices"][0]["message"]
        messages.append(msg)                       # append-verbatim, always
        try:
            data = json.loads(msg["content"])
        except json.JSONDecodeError as exc:
            feedback = (f"PARSE ERROR: {exc}. Reply with the raw JSON object "
                        "only — no prose, no fences.")
        else:
            errors = validate(data, schema)
            if not errors:
                return data, messages, attempt
            feedback = "VALIDATION ERRORS:\n" + "\n".join(f"- {e}" for e in errors)
        messages.append({"role": "user", "content": feedback})
    return None, messages, max_attempts

for i, brief in enumerate(BRIEFS):
    data, _, attempts = extract_with_retry(brief)
    state = f"VALID after {attempts} attempt(s)" if data else f"FAILED after {attempts}"
    print(f"brief {i}: {state}")

In [ ]:
# What the model actually saw: the feedback messages for briefs 1 and 4.
for i in (1, 4):
    _, messages, _ = extract_with_retry(BRIEFS[i])
    print(f"--- brief {i}: feedback the model was shown ---")
    for m in messages:
        if m["role"] == "user" and "ERROR" in m["content"]:
            print("  " + m["content"].replace("\n", "\n  "))
    print()
print("brief 4's error is identical every time — that repetition is schema")
print("telemetry, not a model bug. The loop is telling you the contract is wrong.")

## Experiment 3 — attack the schema

Brief 4 never converges, and the model is *right*: the customer brought an e-bike; the schema says the world contains no e-bikes. The fix is not a better prompt — it's a better contract. Enums are policy.

**Predict first:** after fixing the enum, how many attempts does brief 4 need? Then ask what else you'd challenge in this draft: is `priority` well-defined? Should `estimate_usd` be required? Is a bare `symptoms` array enough?

In [ ]:
# YOUR ATTEMPT — attack the draft schema yourself before opening the solution.
# Brief 4's device is real; the enum says it doesn't exist. Fix the contract.
FIXED_SCHEMA = None   # your edited copy of WORK_ORDER_SCHEMA goes here

In [ ]:
# SOLUTION — one enum edit; the retry loop and the model are untouched.
import copy
FIXED_SCHEMA = copy.deepcopy(WORK_ORDER_SCHEMA)
FIXED_SCHEMA["properties"]["device"]["enum"].append("e-bike")

data, _, attempts = extract_with_retry(BRIEFS[4], schema=FIXED_SCHEMA)
print(f"brief 4 with fixed schema: {'VALID' if data else 'FAILED'} "
      f"after {attempts} attempt(s)")
print("device:", data["device"], "— the model was never wrong; the contract was")

## Experiment 4 — valid ≠ correct

The shop upgrades to constrained decoding: `MOCK_STRICT` emits schema-valid JSON *every time* (against the fixed schema — asserted below). Does that make the work orders right?

**Predict first — blinded:** without running anything below, read the five briefs and write your own expected `device`, `priority`, and `parts_mentioned` per brief in `EXPECTED`. Then run the solution cell to score the model's agreement with you, n/5.

In [ ]:
# YOUR BLINDED LABELS — fill BEFORE running the solution cell.
# No peeking at MOCK_STRICT: a prediction you didn't write down gets retroactively fixed.
EXPECTED = {}   # e.g. {0: {"device": "laptop", "priority": "low",
                #            "parts_mentioned": ["charging port"]}, ...}

In [ ]:
# SOLUTION — schema-valid outputs, then the agreement score.
MOCK_STRICT = {
    0: _wo("Dana", "laptop", "low", ["won't charge", "loose charging port"],
           ["charging port"]),
    1: _wo("Marco", "tablet", "low", ["screen flickers intermittently"], []),
    2: _wo("Priya (Northwind)", "phone", "normal",
           ["office phone dead", "backup console overheats", "swollen battery"],
           ["battery"]),
    3: _wo("Lucía", "laptop", "normal", ["shuts off by itself after ~10 min"],
           ["battery"]),
    4: _wo("Theo", "e-bike", "urgent", ["display goes dark on hills"],
           ["controller", "battery-management board"]),
}

def mock_model_strict(messages):
    brief_idx = next(i for i, b in enumerate(BRIEFS) if b in messages[0]["content"])
    msg = {"role": "assistant", "content": MOCK_STRICT[brief_idx], "tool_calls": []}
    return {"choices": [{"message": msg}], "usage": {"total_tokens": 30}}

REFERENCE_LABELS = {
    0: {"device": "laptop", "priority": "low", "parts_mentioned": ["charging port"]},
    1: {"device": "tablet", "priority": "low", "parts_mentioned": []},
    2: {"device": "phone", "priority": "urgent", "parts_mentioned": ["battery"]},
    3: {"device": "laptop", "priority": "normal", "parts_mentioned": ["battery"]},
    4: {"device": "e-bike", "priority": "urgent", "parts_mentioned": ["controller"]},
}
if not EXPECTED:
    print("(no blinded labels recorded — scoring against the reference labels)\n")
labels = {**REFERENCE_LABELS, **EXPECTED}

agree, misses = 0, []
for i, brief in enumerate(BRIEFS):
    out = json.loads(mock_model_strict([{"role": "user", "content": PROMPT + brief}])
                     ["choices"][0]["message"]["content"])
    assert validate(out, FIXED_SCHEMA) == []      # every output IS schema-valid
    diff = {k: (out[k], v) for k, v in labels[i].items() if out[k] != v}
    agree += not diff
    if diff:
        misses.append((i, diff))

print(f"agreement: {agree}/{len(BRIEFS)}")
for i, diff in misses:
    print(f"  brief {i}, model vs labels: {diff}")
print("\nEvery output validated. Two were still wrong: one in-enum wrong value,")
print("one hallucinated part. Those are the two failure classes no validator")
print("can see — which is what the agreement number is for.")

## What transfers

- `validate` → pydantic or `jsonschema` against your real schema: same checks, more keywords, better error text.
- `extract_with_retry` → the portable fallback every backend supports: validate locally, append the error, re-ask, cap. Libraries like Instructor are this loop with plumbing.
- constrained decoding (Structured Outputs, guided decoding in open runtimes) → removes the *parse* and *schema* failure classes at sampling time. Experiment 4 is why you keep the agreement score anyway.
- brief 4's repeated identical error → schema review as a standing practice: validation errors are telemetry about the contract, not just the model.
- `EXPECTED`, blinded-first → how you audit any generator whose output feeds a decision: labels before outputs, agreement n/5 after.

Next: a validated plan is not an *approved* one. S05 puts a human gate between the spec and its execution.